# **Dati geografici**

Pipeline di elaborazione dei dati riferiti ***dati geografici***

In [ ]:
!pip install pandas sqlalchemy psycopg2-binary xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 7.1 MB/s eta 0:00:00


Esegue il mount del drive di Google dove sono salvati i dataset

In [ ]:
from google.colab import drive
# Smonta il drive per azzerare la cache di sessione
drive.flush_and_unmount()
# Eseguo il mount del drive Google
drive.mount('/content/drive', force_remount=True)
# Flag per indicare se il salvataggio del dataset di produzione deve essere
# eseguito su database oppure su file excel
FLAG_SALVATAGGIO_DB = False

# Configurazione percorso in Colab ove sono presenti i file csv da caricare
CARTELLA_DATI_GEOGRAFICI ='/content/drive/MyDrive/Ricicla(MI)/Dati_geografici/dataset/'
# Nome del file contenente il dataset iniziale
FILE_DATI_GEOGRAFICI = "dati_geografici.csv"

Mounted at /content/drive


Legge il dataset da file e lo carica in un dataframe pandas

In [ ]:
import os as os
import pandas as pd

# Verifica che la cartella esista e carico il dataset
if not os.path.exists(CARTELLA_DATI_GEOGRAFICI):
    print(f"ERRORE: La cartella '{CARTELLA_DATI_GEOGRAFICI}' non esiste.")
else:
  print(f"Leggo il file '{FILE_DATI_GEOGRAFICI}' dalla cartella '{CARTELLA_DATI_GEOGRAFICI}'")
  df = pd.read_csv(CARTELLA_DATI_GEOGRAFICI + FILE_DATI_GEOGRAFICI, sep=";",dtype=str)
  print(f"File '{FILE_DATI_GEOGRAFICI}' caricato correttamente")

Leggo il file 'dati_geografici.csv' dalla cartella '/content/drive/MyDrive/Ricicla(MI)/Dati_geografici/dataset/'
File 'dati_geografici.csv' caricato correttamente


Esegue alcuni controlli sul dataframe per verificare se il dataset è completo

In [ ]:
#data quality
import numpy as np
pd.set_option('future.no_silent_downcasting', True)

# Standardizza i vuoti: trasforma spazi vuoti e stringhe vuote in NaN
df_controllo = df.replace([r'^\s*$', 'None', 'NaN'], np.nan, regex=True)

# Verifica se esiste ALMENO un valore nullo in tutto il DataFrame
if df_controllo.isna().any().any():
  print("ALERT: Ci sono valori vuoti o mancanti all'interno del dataset")

  # Mostra quali colonne contengono i vuoti e quanti sono
  conteggio_vuoti = df_controllo.isna().sum()
  print("\nDettaglio dei vuoti per colonna:")
  print(conteggio_vuoti[conteggio_vuoti > 0])
else:
  print("Il dataset è corretto e non ha valori vuoti.")

ALERT: Ci sono valori vuoti o mancanti all'interno del dataset

Dettaglio dei vuoti per colonna:
comune                     1
sigla_provincia           92
data_inizio_validita       1
data_fine_validita      7894
stato_validita             1
dtype: int64


In [ ]:
# Definisci una funzione personalizzata per standardizzare i valori
def standardize_empty_values(value):
    if isinstance(value, str):
        # Rimuovi spazi bianchi e verifica se la stringa è vuota
        if value.strip() == '':
            return np.nan
        # Verifica se la stringa è 'None' o 'NaN' (case-insensitive)
        elif value.lower() in ['none', 'nan']:
            return np.nan
    return value

# Applica la funzione a ogni elemento del DataFrame
df_controllo_alternativo = df.map(standardize_empty_values)

# Verifica se esiste ALMENO un valore nullo in tutto il DataFrame nel DataFrame alternativo
if df_controllo_alternativo.isna().any().any():
  print("ALERT: Ci sono valori vuoti o mancanti all'interno del dataset (usando map)")
  conteggio_vuoti_alternativo = df_controllo_alternativo.isna().sum()
  print("\nDettaglio dei vuoti per colonna (usando map):")
  print(conteggio_vuoti_alternativo[conteggio_vuoti_alternativo > 0])
else:
  print("Il dataset alternativo è corretto e non ha valori vuoti.")

# Puoi confrontare i due DataFrame se vuoi essere sicuro che i risultati siano gli stessi
# print((df_controllo.compare(df_controllo_alternativo)))

ALERT: Ci sono valori vuoti o mancanti all'interno del dataset (usando map)

Dettaglio dei vuoti per colonna (usando map):
comune                   1
sigla_provincia         92
data_inizio_validita     1
dtype: int64


Sistemo il dataset

In [ ]:
try:
  df = df.drop(columns="data_fine_validita")
  df['stato_validita'] = df['stato_validita'].fillna('Attivo')
  #df['data_inizio_validita'] = pd.to_datetime(df['data_inizio_validita'], format='%Y-%m-%d')
except Exception as e:
  print(f"ERRORE: {e}")

ERRORE: "['data_fine_validita'] not found in axis"


Imposta i parametri di riferimento per la connessione a datase, oppure per il salvataggio su filesystem

In [ ]:
from google.colab import userdata

# Parametri per la configurazione della connesisone al database
DB_USER = "postgres.luhxmgsxvbkfuylgkthr"
DB_PASSWORD = userdata.get("SUPABASE_PASSWORD")
DB_HOST = "aws-0-eu-west-1.pooler.supabase.com"
DB_PORT = "5432"
DB_NAME = "postgres"
DATABASE_URL = (f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

# Nome della tabella di produzione
TAB_DATI_GEOGRAFICI_PROD = "dati_geografici"

# Parametri per il salvataggio su filesystem
CARTELLA_OUTPUT = "/content/drive/MyDrive/Ricicla(MI)/output/"
FILE_DATI_GEOGRAFICI_OUT = "dati_geografici.xlsx"

Salva il dataset elaborato su database oppure su file system

In [ ]:
from sqlalchemy import create_engine

# Creo l'engine tramite la funzione create_engine di SQLAlchemy
if FLAG_SALVATAGGIO_DB:
  try:
    print(f"Creazione della connessione verso 'postgresql://{DB_USER}:***@{DB_HOST}:{DB_PORT}/{DB_NAME}'")
    engine = create_engine(DATABASE_URL)
    print(f"Connessione creata")
    df.to_sql(TAB_DATI_GEOGRAFICI_PROD, con=engine, if_exists="replace", index=False)
    print(f"Dataset salvato sulla tabella '{TAB_DATI_GEOGRAFICI_PROD}'")
  except Exception as e:
    print(f"ERRORE: {e}")
  finally:
    engine.dispose()
else:
  try:
    df.to_excel(CARTELLA_OUTPUT + FILE_DATI_GEOGRAFICI_OUT, index=False)
    print(f"Dati salvati nella cartella '{CARTELLA_OUTPUT}' su file {FILE_DATI_GEOGRAFICI_OUT}")
  except Exception as e:
    print(f"ERRORE: {e}")

Dati salvati nella cartella '/content/drive/MyDrive/Ricicla(MI)/output/' su file dati_geografici.xlsx


Cancella il dataframe

In [ ]:
#del df